# Rakuten Multimodal Product Classification

Fusion of the two single-modality models: EfficientNet-B0 (image) and XLM-RoBERTa (text).
Both pretrained branches stay frozen; only the fusion classifier on top is trained.

Contents:
1. Setup and imports
2. Text preprocessing and data preparation
3. Model definitions (branches + fusion)
4. Datasets and loaders
5. Training (concatenation fusion)
6. Evaluation
7. Test data preparation
8. Cross-attention fusion (alternative)
9. Fusion gain ablation

# 1. Setup and Imports

`torch` and the device are initialised **before** all other imports. On Apple Silicon (MPS) a
different import order can cause segfaults inside Jupyter.

In [ ]:
import os

os.chdir(os.path.expanduser('~/Desktop/Alles/Liora/Rakuten_Project'))  # change to your specific path
BASE_DIR = os.getcwd()
IMAGES_DIR = os.path.join(BASE_DIR, 'images')
DATA_DIR = os.path.join(BASE_DIR, 'text')
IMAGES_CROPPED_DIR = os.path.join(BASE_DIR, 'images', 'cropped_image_train')
IMAGES_CROPPED_DIR_TEST = os.path.join(BASE_DIR, 'images', 'cropped_image_test')

In [ ]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(device)

In [ ]:
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch.nn as nn
from bs4 import BeautifulSoup
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer

# 2. Text Preprocessing and Data Preparation

Text cleaning taken from the text workstream, so the multimodal model sees exactly the same
input as the standalone text model.

In [ ]:
def combine_text_columns(df, text_columns=('designation', 'description')):
    """Combine designation + description into one column.
    description is missing ~35% of the time, designation never -> fillna first."""
    df[text_columns[0]] = df[text_columns[0]].fillna('')
    df[text_columns[1]] = df[text_columns[1]].fillna('')
    df['text'] = df[text_columns[0]] + ' ' + df[text_columns[1]]
    return df


def clean_text(text):
    if not isinstance(text, str):
        return ''

    # Strip HTML tags and decode entities (&eacute;, &amp;, ...).
    # BeautifulSoup handles malformed/nested tags and entity decoding,
    # which a plain `<.*?>` regex cannot do reliably.
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Collapse whitespace only. Do NOT lowercase, strip accents, remove
    # digits/punctuation, or remove stopwords - the model needs them.
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def preprocess_text(df):
    df = combine_text_columns(df)
    df['text'] = df['text'].apply(clean_text)
    return df

In [ ]:
df = pd.read_csv(DATA_DIR + '/X_train_update.csv')
df = preprocess_text(df)

# Build the path to the cropped image from imageid / productid
df['image_path'] = (
    IMAGES_CROPPED_DIR + '/image_' + df['imageid'].astype(str)
    + '_product_' + df['productid'].astype(str) + '.jpg'
)

# Check that every row has an existing image file
df['file_exists'] = df['image_path'].apply(os.path.exists)
print(df['file_exists'].value_counts())
df.drop(columns=['file_exists'], inplace=True)

# Attach the labels
y = pd.read_csv(DATA_DIR + '/Y_train_CVw08PX.csv')
df = df.merge(y, on='Unnamed: 0')
df.head()

# 3. Model Definitions

## Text branch

Same architecture as in the text workstream, so the trained checkpoint can be loaded into it.

In [ ]:
class XLMRoBERTaClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained("xlm-roberta-base")
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.drop(cls)
        return self.classifier(cls)

## Fusion model

The image features (1280-dim, after `global_pool`) and the text features (768-dim, CLS token)
are concatenated and fed into a small classifier. Both pretrained branches are frozen, so only
the fusion head (~1M parameters) is trained.

In [ ]:
class MultimodalClassifier(nn.Module):
    def __init__(self, image_model, text_model, num_classes=27):
        super().__init__()
        self.image_model = image_model
        self.text_model = text_model

        # Freeze both pretrained branches completely
        for param in self.image_model.parameters():
            param.requires_grad = False
        for param in self.text_model.parameters():
            param.requires_grad = False

        image_feature_size = self.image_model.num_features
        text_feature_size = self.text_model.bert.config.hidden_size
        combined_size = image_feature_size + text_feature_size

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_model.forward_features(image)
        image_features = self.image_model.global_pool(image_features)

        text_output = self.text_model.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_output.last_hidden_state[:, 0, :]

        combined = torch.cat([image_features, text_features], dim=1)
        return self.classifier(combined)

## Load the two trained branches

In [ ]:
le = joblib.load('label_encoder.pkl')
num_classes = len(le.classes_)

text_model = XLMRoBERTaClassifier(num_classes).to(device)
text_model.load_state_dict(torch.load('new_xlm_roberta_best_bs.pt',
                                      map_location=torch.device('cpu')))

image_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
image_model.classifier = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
)
image_model.load_state_dict(torch.load('final_image_model.pth'))
image_model = image_model.to(device)

In [ ]:
multimodal_model = MultimodalClassifier(image_model, text_model, num_classes=27).to(device)

trainable = sum(p.numel() for p in multimodal_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in multimodal_model.parameters())
print(f"{trainable:,} out of {total:,} parameters trainable ({100*trainable/total:.1f}%)")

# 4. Datasets and Loaders

In [ ]:
y_encoded = le.transform(df['prdtypecode'])

X_train_img, X_val_img, X_train_text, X_val_text, y_train, y_val = train_test_split(
    df['image_path'], df['text'], y_encoded,
    test_size=0.25, random_state=42, stratify=y_encoded
)

In [ ]:
# timm exposes the preprocessing config the pretrained model was trained with
_reference_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
data_config = timm.data.resolve_data_config({}, model=_reference_model)
del _reference_model
print(data_config)

transform_train = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config['mean'], std=data_config['std'])
])
transform_val = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config['mean'], std=data_config['std'])
])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, image_paths, texts, labels, tokenizer, transform=None, max_length=128):
        self.image_paths = image_paths.tolist()
        self.texts = texts.tolist()
        self.labels = labels
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load the image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Tokenize the text
        text = self.texts[idx]
        encoding = self.tokenizer(
            text, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        label = self.labels[idx]

        return image, input_ids, attention_mask, label

In [ ]:
train_dataset = MultimodalDataset(
    image_paths=X_train_img,
    texts=X_train_text,
    labels=y_train,
    tokenizer=tokenizer,
    transform=transform_train
)

val_dataset = MultimodalDataset(
    image_paths=X_val_img,
    texts=X_val_text,
    labels=y_val,
    tokenizer=tokenizer,
    transform=transform_val
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 5. Training (concatenation fusion)

The frozen backbones are explicitly kept in `eval()` mode inside the training loop: otherwise
BatchNorm in EfficientNet keeps updating its running statistics and dropout in RoBERTa fires,
which silently corrupts the frozen features that are being fused.

Early stopping runs on the validation loss; the checkpoint is saved on the best F1. The two are
independent, since the epoch with the lowest validation loss is not always the best-F1 epoch.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(multimodal_model.parameters(), lr=0.00002)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       factor=0.5, patience=2)

num_epochs = 40
best_val_loss = float('inf')
patience = 5
patience_counter = 0
best_f1 = 0
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'f1': [], 'accuracy': []}

for epoch in range(num_epochs):
    multimodal_model.train()
    multimodal_model.image_model.eval()
    multimodal_model.text_model.eval()

    train_loss = 0
    train_all_labels = []
    train_all_predicted = []

    for images, input_ids, attention_mask, labels in train_loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = multimodal_model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_all_labels.extend(labels.cpu().numpy())
        train_all_predicted.extend(predicted.cpu().numpy())

    train_f1 = f1_score(train_all_labels, train_all_predicted, average='weighted')

    multimodal_model.eval()
    val_loss = 0
    correct = 0
    total_val = 0
    all_labels = []
    all_predicted = []

    with torch.no_grad():
        for images, input_ids, attention_mask, labels in val_loader:
            images = images.to(device)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = multimodal_model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_predicted.extend(predicted.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    avg_train_loss = train_loss / len(train_loader)
    accuracy = correct / total_val
    f1 = f1_score(all_labels, all_predicted, average='weighted')

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f}")

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_f1'].append(train_f1)
    history['f1'].append(f1)
    history['accuracy'].append(accuracy)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early Stopping!")
            break

    scheduler.step(avg_val_loss)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(multimodal_model.state_dict(), 'best_multimodal_model.pth')

# 6. Evaluation

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
train_loss = history['train_loss']
val_loss = history['val_loss']
train_f1 = history['train_f1']
val_f1 = history['f1']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs, train_f1, color='#3498db', marker='o', label='Train F1')
axes[0].plot(epochs, val_f1, color='#3498db', marker='o', linestyle='--', label='Val F1')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score over epochs', pad=20)
axes[0].legend()
benchmark = 0.8144
axes[0].axhline(y=benchmark, color='black', linestyle='--', linewidth=1.5)
axes[0].text(len(epochs) - 0.25, benchmark + 0.005, f'Benchmark: {benchmark:.4f}',
             ha='right', va='bottom', color='black', fontsize=10)
axes[0].set_ylim(0.8, 1)

axes[1].plot(epochs, train_loss, color='#e74c3c', marker='s', label='Train Loss')
axes[1].plot(epochs, val_loss, color='#e74c3c', marker='s', linestyle='--', label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss over epochs', pad=20)
axes[1].legend()

plt.suptitle('Training: multimodal model', y=1.02)
plt.tight_layout()
plt.show()

## Reload the best checkpoint

The architecture is rebuilt from scratch and the saved weights are loaded, so the evaluation
always runs on the best-F1 checkpoint rather than on whatever state the last epoch left behind.

In [ ]:
image_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
image_model.classifier = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
)

text_model = XLMRoBERTaClassifier(num_classes).to(device)

multimodal_model = MultimodalClassifier(image_model, text_model, num_classes=27).to(device)
multimodal_model.load_state_dict(torch.load('Jon_best_multimodal_model_all.pth', map_location=device))

In [ ]:
multimodal_model.eval()
final_labels = []
final_predicted = []

with torch.no_grad():
    for images, input_ids, attention_mask, labels in val_loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        outputs = multimodal_model(images, input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)
        final_labels.extend(labels.cpu().numpy())
        final_predicted.extend(predicted.cpu().numpy())

In [ ]:
cm = confusion_matrix(final_labels, final_predicted)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
class_names = [f"{code}" for code in le.classes_]

plt.figure(figsize=(18, 16))
ax = sns.heatmap(cm_normalized, fmt='.2f', cmap='Reds', vmin=0, vmax=1,
                 xticklabels=class_names, yticklabels=class_names)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=15)

plt.xlabel('Predicted', fontsize=15)
plt.ylabel('True', fontsize=15)
plt.title('Confusion Matrix (% per true class)', fontsize=20)
plt.xticks(rotation=45, ha='center', fontsize=15)
plt.yticks(rotation=0, fontsize=15)
plt.tight_layout()
plt.show()

# 7. Cross-Attention Fusion (alternative)

Instead of concatenating two pooled vectors, the two modalities attend to each other:

- the image is kept as a **sequence of 49 regions** (7x7), not global-pooled
- the text is kept as a **sequence of tokens**, not reduced to the CLS token
- text attends to image regions AND image regions attend to text

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, image_model, text_model, num_classes=27,
                 d=512, n_heads=8, dropout=0.3):
        super().__init__()
        self.image_model = image_model
        self.text_model = text_model

        # Both pretrained branches stay frozen, exactly as before
        for p in self.image_model.parameters():
            p.requires_grad = False
        for p in self.text_model.parameters():
            p.requires_grad = False

        img_dim = self.image_model.num_features               # 1280
        txt_dim = self.text_model.bert.config.hidden_size     # 768

        # Project both modalities into a shared space so Q @ K.T is defined
        self.img_proj = nn.Linear(img_dim, d)
        self.txt_proj = nn.Linear(txt_dim, d)

        # Direction 1: text queries the image ("which region supports this word?")
        self.txt2img_attn = nn.MultiheadAttention(d, n_heads, dropout=dropout,
                                                  batch_first=True)
        self.norm_txt = nn.LayerNorm(d)

        # Direction 2: image queries the text ("which word describes this region?")
        self.img2txt_attn = nn.MultiheadAttention(d, n_heads, dropout=dropout,
                                                  batch_first=True)
        self.norm_img = nn.LayerNorm(d)

        # Both pooled streams concatenated -> classifier
        self.classifier = nn.Sequential(
            nn.Linear(2 * d, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, image, input_ids, attention_mask, return_attention=False):
        # ---- IMAGE: (B, 1280, 7, 7) -> (B, 49, 1280) -> (B, 49, d)
        img = self.image_model.forward_features(image)
        img = img.flatten(2).transpose(1, 2)
        img = self.img_proj(img)

        # ---- TEXT: (B, L, 768) -> (B, L, d)
        txt = self.text_model.bert(input_ids=input_ids,
                                   attention_mask=attention_mask).last_hidden_state
        txt = self.txt_proj(txt)

        # MultiheadAttention expects True = "ignore this position"
        txt_pad_mask = (attention_mask == 0)                  # (B, L)

        # ---- Direction 1: text attends to image regions
        # Query = text, Key/Value = image. Image has no padding -> no mask needed.
        txt_att, w_txt2img = self.txt2img_attn(
            query=txt, key=img, value=img, need_weights=return_attention
        )
        txt = self.norm_txt(txt + txt_att)                    # residual

        # ---- Direction 2: image attends to text tokens
        # Query = image, Key/Value = text -> padding tokens MUST be masked out
        img_att, w_img2txt = self.img2txt_attn(
            query=img, key=txt, value=txt,
            key_padding_mask=txt_pad_mask, need_weights=return_attention
        )
        img = self.norm_img(img + img_att)                    # residual

        # ---- Pooling
        # Text: masked mean over REAL tokens only. A plain .mean(1) would average
        # in up to 127 padding positions and wash out short titles.
        mask = attention_mask.unsqueeze(-1).float()           # (B, L, 1)
        txt_pooled = (txt * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        # Image: plain mean over the 49 regions (no padding here)
        img_pooled = img.mean(dim=1)                          # (B, d)

        combined = torch.cat([txt_pooled, img_pooled], dim=1)  # (B, 2d)
        logits = self.classifier(combined)

        if return_attention:
            # w_txt2img: (B, L_text, 49)  -> for each word: which image region?
            # w_img2txt: (B, 49, L_text)  -> for each region: which word?
            return logits, w_txt2img, w_img2txt
        return logits

In [ ]:
# Load the two trained branches again (fresh, unfused)
image_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
image_model.classifier = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
)
image_model.load_state_dict(torch.load('final_image_model.pth', map_location=device))

text_model = XLMRoBERTaClassifier(num_classes)
text_model.load_state_dict(torch.load('new_xlm_roberta_best_bs.pt',
                                      map_location=torch.device('cpu')))

ca_model = CrossAttentionFusion(image_model, text_model,
                                num_classes=num_classes).to(device)

trainable = sum(p.numel() for p in ca_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ca_model.parameters())
print(f"{trainable:,} of {total:,} trainable ({100*trainable/total:.1f}%)")

# Sanity check: one batch through the model
imgs, ids, mask, labels = next(iter(train_loader))
with torch.no_grad():
    out, w1, w2 = ca_model(imgs.to(device), ids.to(device), mask.to(device),
                           return_attention=True)
print("logits:   ", out.shape)   # expect (32, 27)
print("txt2img:  ", w1.shape)    # expect (32, 128, 49)
print("img2txt:  ", w2.shape)    # expect (32, 49, 128)

In [ ]:
criterion = nn.CrossEntropyLoss()

# ONLY the trainable params. Passing model.parameters() would hand the optimizer
# 283M frozen tensors -- works, but wasteful and confusing.
optimizer = torch.optim.AdamW(
    [p for p in ca_model.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=0.01
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

num_epochs = 30
patience = 5
patience_counter = 0
best_val_loss = float('inf')
best_f1 = 0.0
MODEL_PATH = 'crossattention_model.pth'

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'f1': [], 'accuracy': []}

for epoch in range(num_epochs):
    # ---------------- train
    ca_model.train()
    # The frozen backbones must stay in eval mode: otherwise BatchNorm in
    # EfficientNet keeps updating its running stats and dropout in RoBERTa fires,
    # which silently corrupts the frozen features you are trying to fuse.
    ca_model.image_model.eval()
    ca_model.text_model.eval()

    train_loss = 0.0
    tr_labels, tr_preds = [], []

    for images, input_ids, attention_mask, labels in train_loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = ca_model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in ca_model.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        tr_labels.extend(labels.cpu().numpy())
        tr_preds.extend(predicted.cpu().numpy())

    train_f1 = f1_score(tr_labels, tr_preds, average='weighted')

    # ---------------- validate
    ca_model.eval()
    val_loss = 0.0
    correct = 0
    total_val = 0
    all_labels, all_predicted = [], []

    with torch.no_grad():
        for images, input_ids, attention_mask, labels in val_loader:
            images = images.to(device)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = ca_model(images, input_ids, attention_mask)
            val_loss += criterion(outputs, labels).item()

            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_predicted.extend(predicted.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    accuracy = correct / total_val
    f1 = f1_score(all_labels, all_predicted, average='weighted')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_f1'].append(train_f1)
    history['f1'].append(f1)
    history['accuracy'].append(accuracy)

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f}")

    # Checkpoint on best F1 (independent of early stopping on loss)
    if f1 > best_f1:
        best_f1 = f1
        torch.save(ca_model.state_dict(), MODEL_PATH)

    # Early stopping on val loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early Stopping!")
            break

print(f"\nBest weighted F1: {best_f1:.4f}")

# 8. Fusion Gain Ablation

How much does each modality actually contribute? Both frozen backbones are run **once** and their
features are cached. A fusion head with the identical architecture is then trained three times:

- `both`: image + text
- `text_only`: image features zeroed out
- `image_only`: text features zeroed out

Because the backbones are frozen, caching the features is exact - no information is lost, but each
head trains in seconds instead of hours.

In [ ]:
SEED = 42

# ---------------------------------------------------------------- frozen branches
image_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=27)
image_model.classifier = nn.Sequential(
    nn.Linear(1280, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 27)
)
image_model.load_state_dict(torch.load('image_model.pth', map_location=device))
image_model = image_model.to(device).eval()

text_model = XLMRoBERTaClassifier(num_classes)
text_model.load_state_dict(torch.load('new_xlm_roberta_best_bs.pt',
                                      map_location=torch.device('cpu')))
text_model = text_model.to(device).eval()

# ---------------------------------------------------------------- loaders (NO augmentation)
# transform_val for BOTH splits: features are precomputed once, so a random
# augmentation would just freeze one arbitrary draw into the cache.
# shuffle=False so nothing can silently misalign.
feat_train_ds = MultimodalDataset(X_train_img, X_train_text, y_train,
                                  tokenizer, transform=transform_val)
feat_val_ds = MultimodalDataset(X_val_img, X_val_text, y_val,
                                tokenizer, transform=transform_val)

feat_train_loader = DataLoader(feat_train_ds, batch_size=64, shuffle=False)
feat_val_loader = DataLoader(feat_val_ds, batch_size=64, shuffle=False)


@torch.no_grad()
def extract_features(loader, name):
    """Run both frozen backbones once and cache the features."""
    img_all, txt_all, lab_all = [], [], []
    for i, (images, input_ids, attention_mask, labels) in enumerate(loader):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # exactly the same feature extraction as in MultimodalClassifier.forward
        f_img = image_model.global_pool(image_model.forward_features(images))   # (B, 1280)
        f_txt = text_model.bert(input_ids=input_ids,
                                attention_mask=attention_mask
                                ).last_hidden_state[:, 0, :]                    # (B, 768)

        img_all.append(f_img.cpu())
        txt_all.append(f_txt.cpu())
        lab_all.append(labels)

        if i % 50 == 0:
            print(f"  {name}: batch {i}/{len(loader)}")

    return (torch.cat(img_all), torch.cat(txt_all), torch.cat(lab_all))


print("Extracting train features...")
img_tr, txt_tr, lab_tr = extract_features(feat_train_loader, "train")
print("Extracting val features...")
img_va, txt_va, lab_va = extract_features(feat_val_loader, "val")

# Concatenate in the SAME order as the original model: [image | text]
F_tr = torch.cat([img_tr, txt_tr], dim=1)      # (N_train, 2048)
F_va = torch.cat([img_va, txt_va], dim=1)      # (N_val,   2048)

IMG_SLICE = slice(0, 1280)
TXT_SLICE = slice(1280, 2048)

print("train:", F_tr.shape, "| val:", F_va.shape)

torch.save({'F_tr': F_tr, 'lab_tr': lab_tr, 'F_va': F_va, 'lab_va': lab_va},
           'fusion_features.pt')
print("Cached to fusion_features.pt")

In [ ]:
class FusionHead(nn.Module):
    """Identical architecture to the classifier inside MultimodalClassifier."""
    def __init__(self, in_dim=2048, num_classes=27):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.net(x)


def mask_features(F, mode):
    """Zero out one modality. Returns a COPY -- never modify the cache in place."""
    F = F.clone()
    if mode == 'text_only':
        F[:, IMG_SLICE] = 0.0        # image ablated
    elif mode == 'image_only':
        F[:, TXT_SLICE] = 0.0        # text ablated
    elif mode != 'both':
        raise ValueError(mode)
    return F


def train_head(mode, num_epochs=30, patience=5, lr=2e-5, batch_size=32):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    head = FusionHead(in_dim=2048, num_classes=num_classes).to(device)

    # ---- move everything to the device ONCE (no DataLoader, no per-batch copy)
    Xtr = mask_features(F_tr, mode).float().to(device)
    ytr = lab_tr.to(device)
    Xva = mask_features(F_va, mode).float().to(device)
    yva = lab_va.to(device)

    n = Xtr.size(0)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss, best_f1, counter = float('inf'), 0.0, 0
    best_preds = None

    for epoch in range(num_epochs):
        head.train()
        perm = torch.randperm(n, device=device)          # shuffle by index
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = criterion(head(Xtr[idx]), ytr[idx])
            loss.backward()
            optimizer.step()

        # ---- validation in ONE forward pass (val set fits comfortably)
        head.eval()
        with torch.no_grad():
            out = head(Xva)
            avg_val_loss = criterion(out, yva).item()
            preds = out.argmax(1).cpu().numpy()

        labs = yva.cpu().numpy()
        f1 = f1_score(labs, preds, average='weighted')
        scheduler.step(avg_val_loss)

        if f1 > best_f1:
            best_f1, best_preds = f1, preds

        if avg_val_loss < best_val_loss:
            best_val_loss, counter = avg_val_loss, 0
        else:
            counter += 1
            if counter >= patience:
                break

    print(f"{mode:>12} | best weighted F1 = {best_f1:.4f}  ({epoch+1} epochs)")
    return best_f1, best_preds


results = {}
predictions = {}
for mode in ['both', 'text_only', 'image_only']:
    results[mode], predictions[mode] = train_head(mode)

In [ ]:
f_both, f_text, f_img = results['both'], results['text_only'], results['image_only']

print(f"\n{'='*52}")
print(f"Full multimodal (image + text) : {f_both:.4f}")
print(f"Text only  (image ablated)     : {f_text:.4f}   -> image contributes +{f_both-f_text:.4f}")
print(f"Image only (text ablated)      : {f_img:.4f}   -> text  contributes +{f_both-f_img:.4f}")
print(f"{'='*52}")

abl = pd.DataFrame({
    'config': ['Image only\n(text ablated)', 'Text only\n(image ablated)',
               'Full multimodal'],
    'f1': [f_img, f_text, f_both]
})
colors = ['#3498db', '#3498db', '#e74c3c']

plt.figure(figsize=(9, 5.5))
sns.barplot(x='config', y='f1', data=abl, palette=colors)
sns.despine(left=True)
for i, r in abl.iterrows():
    plt.text(i, r['f1'] + 0.01, f"{r['f1']:.4f}", ha='center',
             va='bottom', fontweight='bold')
plt.ylabel('F1 Score (weighted)')
plt.xlabel('')
plt.ylim(0, 1.0)
plt.title('Fusion Gain Ablation - contribution of each modality', pad=20)
plt.show()

## Sanity check: did the frozen image branch really stay frozen?

Compares the BatchNorm running statistics of the standalone image model with those stored inside
the trained multimodal checkpoint. If they drifted, the backbone was not properly in `eval()` mode
during training.

In [ ]:
sd_img = torch.load('image_model.pth', map_location='cpu')
sd_mm = torch.load('Jon_best_multimodal_model_all.pth', map_location='cpu')

drifts = []
for k in sd_img:
    if 'running_mean' in k or 'running_var' in k:
        a = sd_img[k]
        b = sd_mm['image_model.' + k]
        rel = (a - b).abs().mean() / (a.abs().mean() + 1e-8)
        drifts.append(rel.item())

drifts = torch.tensor(drifts)
print(f"BatchNorm buffers checked : {len(drifts)}")
print(f"mean relative drift       : {drifts.mean():.4f}")
print(f"max  relative drift       : {drifts.max():.4f}")

## Multi-seed run

A single seed says nothing about whether a small gain is real. The whole ablation is repeated over
five seeds, and the per-class F1 is collected as well.

In [ ]:
lab_np = lab_va.numpy()
class_str = le.classes_.astype(str)


def per_class_f1(preds):
    rep = classification_report(lab_np, preds, output_dict=True,
                                zero_division=0, target_names=class_str)
    return pd.Series({c: rep[c]['f1-score'] for c in class_str})


agg = []                                                # aggregate F1 per seed
pc = {'both': [], 'text_only': [], 'image_only': []}    # per-class F1 per seed

for seed in [0, 1, 2, 42, 161]:
    SEED = seed
    row = {'seed': seed}
    for mode in ['both', 'text_only', 'image_only']:
        f1, preds = train_head(mode, num_epochs=150)
        row[mode] = f1
        pc[mode].append(per_class_f1(preds))
    agg.append(row)
    print(f"seed {seed}: both={row['both']:.4f}  text_only={row['text_only']:.4f}  "
          f"image_only={row['image_only']:.4f}  gain={row['both']-row['text_only']:+.4f}")

agg = pd.DataFrame(agg).set_index('seed')
print("\n", agg.agg(['mean', 'std']).to_string(float_format='%.4f'))

In [ ]:
m, s = agg.mean(), agg.std()
order = ['image_only', 'text_only', 'both']
labels = ['Image only\n(text ablated)', 'Text only\n(image ablated)', 'Full multimodal']

plt.figure(figsize=(9, 5.5))
plt.bar(labels, m[order], yerr=s[order], capsize=5,
        color=['#3498db', '#3498db', '#e74c3c'])
sns.despine(left=True)
for i, k in enumerate(order):
    plt.text(i, m[k] + s[k] + 0.015, f"{m[k]:.4f}", ha='center', fontweight='bold')
plt.ylabel('F1 Score (weighted)')
plt.ylim(0, 1.0)
plt.title('Fusion Gain Ablation (mean +/- SD over 5 seeds)', pad=20)
plt.show()

In [ ]:
both_df = pd.DataFrame(pc['both'])          # 5 seeds x 27 classes
text_df = pd.DataFrame(pc['text_only'])
gain_df = both_df - text_df                 # per-seed gain

d = pd.DataFrame({
    'gain': gain_df.mean(),
    'sd': gain_df.std(),
}).sort_values('gain', ascending=False)

print(d['gain'].to_string(float_format='%.4f'))

# red = gain clearly outside its own seed variation, grey = not distinguishable
colors = ['#e74c3c' if abs(g) > 2 * s else '#bdc3c7'
          for g, s in zip(d['gain'], d['sd'])]

plt.figure(figsize=(9, 8))
plt.barh(range(len(d)), d['gain'], xerr=d['sd'], color=colors,
         error_kw={'ecolor': '#34495e', 'capsize': 3, 'lw': 1})
plt.yticks(range(len(d)), d.index, fontsize=9)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('F1 gain from the image branch (mean +/- SD over 5 seeds)')
plt.ylabel('prdtypecode')
plt.title('Per-class contribution of the image modality', pad=20)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()